# 📘 Lab TP — Construire un Chatbot RAG 100 % Local - End to End

- À la fin de ce TP, l’étudiant sera capable de :
    -   Comprendre quand utiliser (ou non) un RAG
    -   Implémenter un RAG complet 100 % local
    -   Gérer sécurité (RBAC), performance, observabilité
    -   Comparer qualité vs latence
    -   Déployer un chatbot RAG via une API Flask

# A- Mise en contexte & données

### 🎯 Contexte
---

-   Fashion Forward Hub est une boutique de vêtements en ligne qui souhaite améliorer l’expérience client grâce à un assistant intelligent capable de :
    -   répondre aux questions fréquentes,
    -   fournir des informations sur les produits,
    -   aider à choisir des articles ou composer des looks.
Dans ce TP, nous allons construire cet assistant pas à pas, en appliquant les principes du Retrieval-Augmented Generation (RAG).


-   Nous allons volontairement retarder l’usage du RAG vectoriel, afin de comprendre pourquoi et quand il est réellement utile.

### 1-Comprendre le schéma de données de Fashion Forward Hub
---

Fashion Forward Hub stocke ses informations dans **deux bases de données distinctes** :

- **Base Produits** : contient l’ensemble des produits et leurs caractéristiques.
- **Base FAQ** : contient les questions fréquentes et leurs réponses.

Avant de construire un système de type **RAG (Retrieval-Augmented Generation)**, il est essentiel de bien comprendre :
- quelles informations sont disponibles,
- comment elles sont structurées,
- et comment elles pourront être utilisées pour répondre aux questions des utilisateurs.

Nous commençons par explorer la **base de données Produits**.


Les attributs disponibles pour chaque produit sont les suivants :

- **Genre** : Public cible du produit, par exemple « Men », « Women » ou « Unisex ».
- **Catégorie principale** (*Master Category*) : Classification large du produit, comme « Apparel » (vêtements) ou « Footwear » (chaussures).
- **Sous-catégorie** (*Sub Category*) : Catégorie plus spécifique à l’intérieur de la catégorie principale, par exemple « Topwear ».
- **Type d’article** (*Article Type*) : Type exact du produit, par exemple « Shirts » (chemises) ou « Jackets » (vestes).
- **Couleur principale** (*Base Colour*) : Couleur dominante du produit, un critère important pour le choix du client.
- **Saison** (*Season*) : Saison pour laquelle le produit est conçu, par exemple « Summer » (été) ou « Winter » (hiver).
- **Année** (*Year*) : Année de sortie ou de la collection du produit.
- **Usage** (*Usage*) : Usage ou occasion prévue pour le produit, comme « Casual » (décontracté) ou « Formal » (formel).
- **Nom d’affichage du produit** (*Product Display Name*) : Nom descriptif utilisé à des fins marketing.
- **Prix** (*Price*) : Coût du produit.
- **Identifiant du produit** (*Product ID*) : Identifiant unique permettant de gérer et de suivre le produit dans le catalogue.


In [1]:
import os, json, time
import numpy as np
import joblib
from sklearn.neighbors import NearestNeighbors

import utils

print("✅ utils importé depuis :", utils.__file__)
print("✅ OLLAMA_URL =", utils.OLLAMA_URL)

# Vérifier que les fichiers existent
for p in ["data/clothes_json.joblib", "data/faq.joblib", "data/clothes.csv"]:
    print(p, "->", "OK" if os.path.exists(p) else "ABSENT")

✅ utils importé depuis : /Users/macbookpro/Desktop/Teaching/generativeai_class/lab4/full-rag/utils.py
✅ OLLAMA_URL = http://localhost:11434
data/clothes_json.joblib -> OK
data/faq.joblib -> OK
data/clothes.csv -> OK


#### A.1- Base Produits — Schéma et exploration
---
-   comprendre comment les produits sont stockés et décrits.

In [2]:
import joblib

# Chargement des données produits
PRODUCTS_DATA = joblib.load("data/clothes_json.joblib")

print("Nombre total de produits :", len(PRODUCTS_DATA))
print("Type d’un enregistrement :", type(PRODUCTS_DATA[0]))

# Inspection d’un produit exemple
example_product = PRODUCTS_DATA[0]

print("\n🧩 Champs disponibles (schéma) :")
print(list(example_product.keys()))

print("\n🔎 Exemple de produit (aperçu des premiers champs) :")
for key in list(example_product.keys())[:12]:
    print(f"- {key} : {example_product.get(key)}")

Nombre total de produits : 44424
Type d’un enregistrement : <class 'dict'>

🧩 Champs disponibles (schéma) :
['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'price', 'product_id']

🔎 Exemple de produit (aperçu des premiers champs) :
- gender : Men
- masterCategory : Apparel
- subCategory : Topwear
- articleType : Shirts
- baseColour : Navy Blue
- season : Fall
- year : 2011.0
- usage : Casual
- productDisplayName : Turtle Check Men Navy Blue Shirt
- price : 67
- product_id : 15970


- Chaque produit est représenté par un **objet JSON structuré**.
- La base contient :
  - des **attributs catégoriels** (genre, catégorie, couleur, saison, usage),
  - des **attributs descriptifs** (nom du produit),
  - et des **attributs numériques** (prix, année).
- Ces informations devront être **transformées en texte** afin d’être comprises
  par un modèle de langage.

Une bonne compréhension du schéma de données est indispensable avant de :
- calculer des embeddings,
- construire un index vectoriel,
- ou répondre aux questions des utilisateurs avec un système RAG.


#### A.2- Base FAQ — Structure et usage
---

- comprendre la structure des FAQ.

In [3]:
import joblib

# Chargement de la base FAQ
FAQ_DATA = joblib.load("data/faq.joblib")

print("✅ Nombre total de questions FAQ :", len(FAQ_DATA))
print("✅ Type d’un élément FAQ :", type(FAQ_DATA[0]))

print("\n🔎 Exemple de FAQ :")
FAQ_DATA[0]


✅ Nombre total de questions FAQ : 25
✅ Type d’un élément FAQ : <class 'dict'>

🔎 Exemple de FAQ :


{'question': 'What are your store hours?',
 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
 'type': 'general information'}

- Les FAQ sont stockées comme une liste de dictionnaires contenant : `question`, `answer`, `type`

Cette base contient des informations générales telles que :
- les politiques de livraison,
- les retours et remboursements,
- les tailles,
- ou d’autres questions courantes posées par les clients.

Contrairement à la base Produits, la FAQ :
- ne décrit pas des articles spécifiques,
- mais fournit des **réponses génériques** utiles pour améliorer l’expérience utilisateur.

Nous allons maintenant explorer sa structure.


In [4]:
import pandas as pd

df_faq = pd.DataFrame(FAQ_DATA)

print("📐 Dimensions du DataFrame FAQ :", df_faq.shape)
print("\n🧾 Colonnes disponibles dans la FAQ :")
print(list(df_faq.columns))

print("\n🎯 Aperçu des premières entrées :")
df_faq.head(5)


📐 Dimensions du DataFrame FAQ : (25, 3)

🧾 Colonnes disponibles dans la FAQ :
['question', 'answer', 'type']

🎯 Aperçu des premières entrées :


,question,answer,type
0,What are your store hours?,Our online store is open 24/7. Customer servic...,general information
1,Where is Fashion Forward Hub located?,Fashion Forward Hub is primarily an online sto...,general information
2,Do you have a physical store location?,"At this time, we operate exclusively online. T...",general information
3,How can I create an account with Fashion Forwa...,Click on 'Sign Up' in the top right corner of ...,general information
4,How do I subscribe to your newsletter?,"To receive the latest updates and promotions, ...",general information


- La base FAQ est constituée d’une **liste de paires question / réponse**.
- Chaque entrée contient généralement :
  - une **question** formulée en langage naturel,
  - une **réponse textuelle** prête à être affichée à l’utilisateur.
- Ces données sont particulièrement adaptées pour :
  - répondre aux questions générales,
  - compléter les réponses issues de la base Produits.

Dans ce projet, la FAQ sera utilisée comme **contexte textuel** intégré directement
dans le prompt, plutôt que comme une base indexée séparément.


- Nous avons maintenant une vision claire des deux sources d’information : les produits et la FAQ.
- L’étape suivante consiste à transformer ces données en documents exploitables par un système RAG.

#### A.3- Vue d'ensemble des deux bases

In [5]:
PRODUCTS = PRODUCTS_DATA
FAQ = FAQ_DATA

print(" Nb produits :", len(PRODUCTS))
print(" Nb FAQ :", len(FAQ))

print("Exemple produit keys:", list(PRODUCTS[0].keys())[:12])
print("Exemple FAQ:", FAQ[0])

 Nb produits : 44424
 Nb FAQ : 25
Exemple produit keys: ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'price', 'product_id']
Exemple FAQ: {'question': 'What are your store hours?', 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.', 'type': 'general information'}


# B- Raisonnement avant le RAG

- Un bon système RAG commence par du raisonnement, pas par des embeddings.

#### B.1- Task Routing — FAQ ou Produit ?
---
- déterminer la nature générale de la requête utilisateur.

In [6]:
import re

FAQ_KEYWORDS = [
    "livraison", "retour", "remboursement", "paiement", "commande", "annuler",
    "échange", "guide des tailles", "taille", "expédition", "délais", "frais",
    "service client", "compte", "mot de passe", "suivi", "adresse"
]

PRODUCT_KEYWORDS = [
    "je cherche", "je veux", "propose", "montre", "chemise", "jean", "t-shirt", "tshirt",
    "chaussure", "couleur", "taille", "hiver", "été", "casual", "formal",
    "tenue", "look", "assortir", "outfit", "style"
]

def route_query(query: str) -> str:
    q = query.lower()
    faq_score = sum(1 for w in FAQ_KEYWORDS if w in q)
    prod_score = sum(1 for w in PRODUCT_KEYWORDS if w in q)

    # règle simple et transparente
    if faq_score > prod_score:
        return "FAQ"
    else:
        return "PRODUCT"

tests = [
    "Quels sont les délais de livraison ?",
    "Comment faire un retour ?",
    "Je cherche une chemise bleu marine pour homme",
    "Propose un look pour une soirée"
]

for t in tests:
    print(route_query(t), "->", t)

FAQ -> Quels sont les délais de livraison ?
FAQ -> Comment faire un retour ?
PRODUCT -> Je cherche une chemise bleu marine pour homme
PRODUCT -> Propose un look pour une soirée


-   `🟦 À retenir`
    -   Toutes les requêtes ne doivent pas aller vers le même pipeline
    -   Le routage réduit les erreurs et le coût

#### B.2- Nature d’une requête produit
--- 
- affiner le type de demande produit. 

- Classer la requête en 3 types:
    - SEARCH : trouver des produits (ex: “je cherche une chemise bleue…”).
    - LOOK : créer une tenue / recommander un style (ex: “propose un look pour…”).
    - COMPARE : comparer des produits / options (ex: “quelle différence entre jeans et chinos ?”, “compare A vs B”).

In [7]:
import re

SEARCH_PATTERNS = [
    r"\bje cherche\b", r"\bje veux\b", r"\bavez[- ]vous\b", r"\bdisponible\b",
    r"\btrouve\b", r"\bchercher\b", r"\bmontre\b", r"\bchemise\b", r"\bjean\b",
    r"\bchaussure\b", r"\bshirt\b", r"\bwatch\b"
]

LOOK_PATTERNS = [
    r"\bpropose\b", r"\bcrée\b", r"\bcompose\b", r"\btenue\b", r"\blook\b",
    r"\boutfit\b", r"\bassortir\b", r"\bstyle\b", r"\bpour\b.*\bmariage\b",
    r"\bpour\b.*\bsoirée\b", r"\bpour\b.*\bbureau\b"
]

COMPARE_PATTERNS = [
    r"\bcompare\b", r"\bcompar(e|aison)\b", r"\bversus\b", r"\bvs\b",
    r"\bdifférence\b", r"\bmeilleur\b.*\bentre\b", r"\bplutôt\b.*\bou\b"
]

def product_task_type(query: str) -> str:
    q = query.lower()

    score_search = sum(1 for p in SEARCH_PATTERNS if re.search(p, q))
    score_look = sum(1 for p in LOOK_PATTERNS if re.search(p, q))
    score_compare = sum(1 for p in COMPARE_PATTERNS if re.search(p, q))

    scores = {"SEARCH": score_search, "LOOK": score_look, "COMPARE": score_compare}
    best = max(scores, key=scores.get)

    # Si tout est 0 -> par défaut SEARCH (le plus fréquent)
    if scores[best] == 0:
        return "SEARCH"
    return best

# Tests pédagogiques
tests = [
    "Je cherche une chemise bleu marine pour homme",
    "Propose un look pour une soirée",
    "Quelle différence entre un jean et un chino ?",
    "Compare deux montres pour femme",
    "Je veux des chaussures noires pour l'hiver"
]

for t in tests:
    print(product_task_type(t), "->", t)


SEARCH -> Je cherche une chemise bleu marine pour homme
LOOK -> Propose un look pour une soirée
SEARCH -> Quelle différence entre un jean et un chino ?
COMPARE -> Compare deux montres pour femme
SEARCH -> Je veux des chaussures noires pour l'hiver


#### B.3- Format d'une requête FAQ.

- Construire le “FAQ layout”

In [8]:
FAQ_DATA = FAQ 

def build_faq_layout(faq_data, max_items=None) -> str:
    lines = []
    items = faq_data if max_items is None else faq_data[:max_items]
    for i, item in enumerate(items, start=1):
        q = (item.get("question", "") or "").strip()
        a = (item.get("answer", "") or "").strip()
        lines.append(f"FAQ {i}\nQ: {q}\nA: {a}\n")
    return "\n".join(lines)

FAQ_LAYOUT = build_faq_layout(FAQ_DATA)
print("✅ FAQ entries:", len(FAQ_DATA))
print("📄 FAQ_LAYOUT preview:\n")
print(FAQ_LAYOUT[:900])


✅ FAQ entries: 25
📄 FAQ_LAYOUT preview:

FAQ 1
Q: What are your store hours?
A: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.

FAQ 2
Q: Where is Fashion Forward Hub located?
A: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State.

FAQ 3
Q: Do you have a physical store location?
A: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you.

FAQ 4
Q: How can I create an account with Fashion Forward Hub?
A: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account.

FAQ 5
Q: How do I subscribe to your newsletter?
A: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage.

FAQ 6
Q: Can I place an order over the phone?
A: Currently, we o


-   `🟦 À retenir`
    -   Un RAG n’est pas toujours adapté aux requêtes créatives
    -   Identifier l’intention améliore fortement la réponse

- Fonction answer_faq()

In [ ]:
FAQ_SYSTEM = (
    "Tu es un assistant du service client de Fashion Forward Hub. "
    "Réponds uniquement à partir des informations contenues dans la FAQ fournie. "
    "Si la réponse n'est pas dans la FAQ, réponds exactement : "
    "Je n'ai pas cette information dans la FAQ."
)

def answer_faq(query: str, model: str = "llama3:latest", temperature: float = 0.0) -> dict:
    user_prompt = (
        f"FAQ:\n{FAQ_LAYOUT}\n\n"
        f"Question utilisateur: {query}\n\n"
        "Réponds en français. Sois clair et concis."
    )

    out = utils.ollama_chat(
        [{"role": "system", "content": FAQ_SYSTEM},
         {"role": "user", "content": user_prompt}],
        model=model,
        temperature=temperature
    )
    return {"task": "FAQ", "query": query, "answer": out.text, "model": model, "latency_ms": out.latency_ms}

# Test
res_faq = answer_faq("Quels sont les délais de livraison ?")
print(res_faq["answer"])
print("latency:", res_faq["latency_ms"], "ms")


#### B.4- Extraction de paramètres (Metadata Inference)
--- 
- extraire automatiquement les filtres implicites dans la requête.

In [9]:
import re

# Listes simples (tu pourras enrichir)
GENDER_MAP = {
    "homme": "Men", "men": "Men", "masculin": "Men",
    "femme": "Women", "women": "Women", "féminin": "Women",
    "unisex": "Unisex", "mixte": "Unisex"
}

SEASON_MAP = {
    "été": "Summer", "summer": "Summer",
    "hiver": "Winter", "winter": "Winter",
    "printemps": "Spring", "spring": "Spring",
    "automne": "Fall", "fall": "Fall"
}

USAGE_MAP = {
    "casual": "Casual", "décontracté": "Casual",
    "formal": "Formal", "formel": "Formal",
    "sport": "Sports", "sports": "Sports",
    "party": "Party", "soirée": "Party"
}

# Quelques types produits (à enrichir au fur et à mesure)
ARTICLE_TYPES = [
    ("chemise", "Shirts"),
    ("shirt", "Shirts"),
    ("tshirt", "Tshirts"),
    ("t-shirt", "Tshirts"),
    ("jean", "Jeans"),
    ("montre", "Watches"),
    ("watch", "Watches"),
    ("chaussure", "Casual Shoes"),
    ("shoes", "Casual Shoes"),
    ("socks", "Socks"),
    ("ceinture", "Belts"),
    ("sac", "Handbags"),
]

# Couleurs (à enrichir)
COLOR_MAP = {
    "noir": "Black", "black": "Black",
    "bleu": "Blue", "blue": "Blue",
    "bleu marine": "Navy Blue", "navy": "Navy Blue", "navy blue": "Navy Blue",
    "gris": "Grey", "grey": "Grey",
    "vert": "Green", "green": "Green",
    "violet": "Purple", "purple": "Purple",
    "argent": "Silver", "silver": "Silver",
}

def extract_budget(query: str):
    """
    Extrait un budget max si présent.
    Ex: 'under 50', 'moins de 100', '< 30', 'budget 40'
    """
    q = query.lower()
    # under 50 / less than 50
    m = re.search(r"\b(under|less than)\s+(\d+)\b", q)
    if m:
        return int(m.group(2))

    # moins de 100 / budget 40 / max 30
    m = re.search(r"\b(moins de|budget|max)\s+(\d+)\b", q)
    if m:
        return int(m.group(2))

    # < 30
    m = re.search(r"<\s*(\d+)", q)
    if m:
        return int(m.group(1))

    return None

def extract_params(query: str) -> dict:
    q = query.lower()

    # 1) task type
    task = product_task_type(query)

    # 2) gender
    gender = None
    for k, v in GENDER_MAP.items():
        if k in q:
            gender = v
            break

    # 3) season
    season = None
    for k, v in SEASON_MAP.items():
        if k in q:
            season = v
            break

    # 4) usage
    usage = None
    for k, v in USAGE_MAP.items():
        if k in q:
            usage = v
            break

    # 5) articleType
    article_type = None
    for k, v in ARTICLE_TYPES:
        if re.search(rf"\b{k}\b", q):
            article_type = v
            break

    # 6) baseColour (attention: "bleu marine" doit être testé avant "bleu")
    base_colour = None
    # tri des clés par longueur décroissante pour capter "bleu marine" avant "bleu"
    for k in sorted(COLOR_MAP.keys(), key=len, reverse=True):
        if k in q:
            base_colour = COLOR_MAP[k]
            break

    # 7) budget
    budget_max = extract_budget(query)

    return {
        "task_type": task,
        "filters": {
            "gender": gender,
            "season": season,
            "usage": usage,
            "articleType": article_type,
            "baseColour": base_colour,
            "price_max": budget_max,
        }
    }

# Tests
tests = [
    "Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50",
    "Propose un look pour une soirée (femme), couleur noir, budget 100",
    "Any very cheap items under 5 dollars?",
    "Compare deux montres argentées pour femme"
]

for t in tests:
    print(t)
    print(extract_params(t))
    print("-"*60)


Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50
{'task_type': 'SEARCH', 'filters': {'gender': 'Men', 'season': 'Winter', 'usage': 'Casual', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'price_max': 50}}
------------------------------------------------------------
Propose un look pour une soirée (femme), couleur noir, budget 100
{'task_type': 'LOOK', 'filters': {'gender': 'Women', 'season': None, 'usage': 'Party', 'articleType': None, 'baseColour': 'Black', 'price_max': 100}}
------------------------------------------------------------
Any very cheap items under 5 dollars?
{'task_type': 'SEARCH', 'filters': {'gender': None, 'season': None, 'usage': None, 'articleType': None, 'baseColour': None, 'price_max': 5}}
------------------------------------------------------------
Compare deux montres argentées pour femme
{'task_type': 'COMPARE', 'filters': {'gender': 'Women', 'season': None, 'usage': None, 'articleType': None, 'baseColour': 'Silver', 'price_max'

-   `🟦 À retenir`
    -   Les métadonnées permettent souvent de résoudre le problème sans RAG
    -   C’est une étape clé dans les systèmes hybrides (LLM + règles)

#### B.5 - Filtrer produits par metadata (sans embeddings)
---
- récupérer des produits candidats via filtrage structuré.

In [12]:
import pandas as pd


def filter_products(df: pd.DataFrame, filters: dict, top_n: int = 10) -> pd.DataFrame:
    out = df.copy()

    # champs exact match
    for col in ["gender", "season", "usage", "articleType", "baseColour"]:
        val = filters.get(col)
        if val is not None and col in out.columns:
            out = out[out[col] == val]

    # prix max
    price_max = filters.get("price_max")
    if price_max is not None and "price" in out.columns:
        # certains datasets ont price en str -> on convertit
        out = out.copy()
        out["price_num"] = pd.to_numeric(out["price"], errors="coerce")
        out = out[out["price_num"].notna()]
        out = out[out["price_num"] <= float(price_max)]

        # petit tri : moins cher d’abord
        out = out.sort_values("price_num", ascending=True)

    # colonnes utiles (si présentes)
    cols = [c for c in ["product_id","id","productDisplayName","gender","articleType","baseColour","season","usage","price"] if c in out.columns]
    return out[cols].head(top_n)



-  Test sur ta requête 1

In [11]:
df_products = pd.DataFrame(PRODUCTS)
query = "Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50"
params = extract_params(query)
candidates = filter_products(df_products, params["filters"], top_n=10)

print("✅ Filtres:", params["filters"])
print("✅ Nb candidats:", len(candidates))
candidates

✅ Filtres: {'gender': 'Men', 'season': 'Winter', 'usage': 'Casual', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'price_max': 50}
✅ Nb candidats: 2


,product_id,productDisplayName,gender,articleType,baseColour,season,usage,price
36117,17187,U.S. Polo Assn. Men Checks Navy Blue Shirt,Men,Shirts,Navy Blue,Winter,Casual,16
6056,17190,U.S. Polo Assn. Men Checks Navy Blue Shirt,Men,Shirts,Navy Blue,Winter,Casual,49


#### B.6- Generate metadata + stratégie de filtrage (relaxation)

In [13]:
from copy import deepcopy

# 1) Génération de metadata standardisée
def generate_metadata(query: str) -> dict:
    parsed = extract_params(query)
    filters = parsed["filters"]

    # On définit une priorité pédagogique :
    # "articleType" et "gender" sont souvent essentiels pour un produit.
    required = ["articleType", "gender"]
    optional = ["baseColour", "season", "usage", "price_max"]

    return {
        "query": query,
        "task_type": parsed["task_type"],
        "filters": filters,
        "required": required,
        "optional": optional,
    }

# 2) Stratégie de relaxation (si zéro résultat)
# On enlève dans cet ordre: couleur -> saison -> usage -> budget -> gender -> articleType
RELAX_ORDER = ["baseColour", "season", "usage", "price_max", "gender", "articleType"]

def relax_filters(filters: dict, relax_key: str) -> dict:
    new_filters = deepcopy(filters)
    new_filters[relax_key] = None
    return new_filters

def retrieve_with_relaxation(df, filters: dict, top_n: int = 10):
    """
    Essaie filtres stricts, sinon relâche progressivement jusqu'à obtenir des résultats.
    Retourne (candidates_df, applied_filters, relaxed_steps)
    """
    steps = []
    current = deepcopy(filters)

    # 1) essai strict
    cand = filter_products(df, current, top_n=top_n)
    if len(cand) > 0:
        return cand, current, steps

    # 2) relaxation progressive
    for key in RELAX_ORDER:
        if current.get(key) is not None:
            current = relax_filters(current, key)
            steps.append(key)
            cand = filter_products(df, current, top_n=top_n)
            if len(cand) > 0:
                return cand, current, steps

    return cand, current, steps

- Teste une requête “trop stricte” pour voir la relaxation

In [14]:
query_strict = "Je cherche une chemise violet pour homme en hiver, budget 5"
meta = generate_metadata(query_strict)
cand, used_filters, relaxed = retrieve_with_relaxation(df_products, meta["filters"], top_n=10)

print("Relaxed steps:", relaxed)
print("Used filters:", used_filters)
print("Nb candidats:", len(cand))
cand.head(5)

Relaxed steps: ['baseColour', 'season', 'price_max']
Used filters: {'gender': 'Men', 'season': None, 'usage': None, 'articleType': 'Shirts', 'baseColour': None, 'price_max': None}
Nb candidats: 10


,product_id,productDisplayName,gender,articleType,baseColour,season,usage,price
0,15970,Turtle Check Men Navy Blue Shirt,Men,Shirts,Navy Blue,Fall,Casual,67
6,30805,Fabindia Men Striped Green Shirt,Men,Shirts,Green,Summer,Ethnic,62
15,12369,Reid & Taylor Men Check Purple Shirts,Men,Shirts,Purple,Fall,Formal,268
30,37812,John Players Men Navy Blue Shirt,Men,Shirts,Navy Blue,Summer,Formal,120
32,56825,John Players Men Brown Shirt,Men,Shirts,Brown,Summer,Casual,62


#### B.7- Générer un contexte produit (sans RAG vectoriel)
---
- transformer les produits filtrés en texte lisible par un LLM.

In [15]:
def format_product_row(row: dict) -> str:
    # On gère les deux cas: id ou product_id selon le dataset
    pid = row.get("product_id", row.get("id", "NA"))
    name = row.get("productDisplayName", "")
    gender = row.get("gender", "")
    art = row.get("articleType", "")
    color = row.get("baseColour", "")
    season = row.get("season", "")
    usage = row.get("usage", "")
    price = row.get("price", row.get("price_num", ""))

    return (
        f"[prod:{pid}]\n"
        f"Name: {name}\n"
        f"Gender: {gender}\n"
        f"ArticleType: {art}\n"
        f"BaseColour: {color}\n"
        f"Season: {season}\n"
        f"Usage: {usage}\n"
        f"Price: {price}\n"
    )

def build_products_context(candidates_df, max_items: int = 8) -> str:
    if candidates_df is None or len(candidates_df) == 0:
        return "No products found."

    rows = candidates_df.head(max_items).to_dict(orient="records")
    blocks = [format_product_row(r) for r in rows]
    return "\n".join(blocks)

- --- Test avec tes candidats de step 5/6 ---


In [16]:
query = "Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50"
meta = generate_metadata(query)

candidates, used_filters, relaxed_steps = retrieve_with_relaxation(df_products, meta["filters"], top_n=10)
products_context = build_products_context(candidates, max_items=8)

print("Relaxed steps:", relaxed_steps)
print("\n--- PRODUCTS CONTEXT (preview) ---\n")
print(products_context[:1200])

Relaxed steps: []

--- PRODUCTS CONTEXT (preview) ---

[prod:17187]
Name: U.S. Polo Assn. Men Checks Navy Blue Shirt
Gender: Men
ArticleType: Shirts
BaseColour: Navy Blue
Season: Winter
Usage: Casual
Price: 16

[prod:17190]
Name: U.S. Polo Assn. Men Checks Navy Blue Shirt
Gender: Men
ArticleType: Shirts
BaseColour: Navy Blue
Season: Winter
Usage: Casual
Price: 49



#### B.8- Génération à partir du contexte produits.
---
- générer une réponse à partir du contexte produit.

In [17]:
PRODUCTS_SYSTEM = (
    "Tu es un assistant shopping pour Fashion Forward Hub. "
    "Réponds UNIQUEMENT en utilisant le contexte PRODUITS fourni. "
    "Chaque phrase factuelle DOIT contenir une citation [prod:...] correspondant au produit utilisé. "
    "Si le contexte ne suffit pas, réponds exactement : "
    "Je n'ai pas assez d'informations dans les produits fournis."
)

def answer_products(query: str, products_context: str, model: str = "llama3:latest", temperature: float = 0.2) -> dict:
    user_prompt = (
        f"Contexte PRODUITS:\n{products_context}\n\n"
        f"Question utilisateur: {query}\n\n"
        "Réponds en français.\n"
        "- Propose au maximum 3 produits.\n"
        "- Pour chaque produit, cite l'ID [prod:...].\n"
        "- Ne mentionne aucun produit qui n'est pas dans le contexte."
    )

    out = utils.ollama_chat(
        [{"role": "system", "content": PRODUCTS_SYSTEM},
         {"role": "user", "content": user_prompt}],
        model=model,
        temperature=temperature
    )

    return {
        "task": "PRODUCT",
        "query": query,
        "model": model,
        "answer": out.text,
        "llm_latency_ms": out.latency_ms,
        "context": products_context,
    }

-  --- Test end-to-end (filters -> candidates -> context -> answer) ---


In [18]:
query = "Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50"

meta = generate_metadata(query)
candidates, used_filters, relaxed_steps = retrieve_with_relaxation(df_products, meta["filters"], top_n=10)
products_context = build_products_context(candidates, max_items=8)

res_prod = answer_products(query, products_context, model="llama3:latest")

print("Relaxed steps:", relaxed_steps)
print(res_prod["answer"])
print("\nllm latency:", res_prod["llm_latency_ms"], "ms")

Relaxed steps: []
Je vous recommande les chemises suivantes pour homme, bleu marine, casual et hiver, avec un budget de 50 :

1. U.S. Polo Assn. Men Checks Navy Blue Shirt [prod:17187] - Prix : 16
C'est une option économique qui correspondra à vos attentes.
2. U.S. Polo Assn. Men Checks Navy Blue Shirt [prod:17190] - Prix : 49
Si vous êtes prêt à investir un peu plus, ce modèle est idéal pour vous.

Je n'ai pas d'autres suggestions dans le contexte fourni.

llm latency: 7951 ms


-   `🟦 À retenir`
    -   Beaucoup de cas réels ne nécessitent pas de RAG vectoriel
    -   Toujours commencer simple

#### B.9- LA fonction finale (router global)

In [ ]:
def assistant(query: str, model: str = "llama3:latest") -> dict:
    # 1) Router FAQ vs PRODUCT
    high_task = route_query(query)

    if high_task == "FAQ":
        r = answer_faq(query, model=model)
        return {"route": "FAQ", **r}

    # 2) Product-related → nature
    task_type = product_task_type(query)

    # 3) SEARCH → pipeline metadata filtering
    if task_type == "SEARCH":
        meta = generate_metadata(query)
        candidates, used_filters, relaxed_steps = retrieve_with_relaxation(df_products, meta["filters"], top_n=10)
        ctx = build_products_context(candidates, max_items=8)

        r = answer_products(query, ctx, model=model)
        r["task_type"] = "SEARCH"
        r["used_filters"] = used_filters
        r["relaxed_steps"] = relaxed_steps
        r["n_candidates"] = 0 if candidates is None else len(candidates)
        return {"route": "PRODUCT", **r}

    # 4) LOOK (à venir)
    if task_type == "LOOK":
        return {
            "route": "PRODUCT",
            "task_type": "LOOK",
            "query": query,
            "answer": "➡️ (À venir) Je vais composer un look en utilisant des produits du catalogue."
        }

    # 5) COMPARE (à venir)
    if task_type == "COMPARE":
        return {
            "route": "PRODUCT",
            "task_type": "COMPARE",
            "query": query,
            "answer": "➡️ (À venir) Je vais comparer des produits/choix à partir du catalogue."
        }

    # fallback
    return {"route": "UNKNOWN", "query": query, "answer": "Je ne suis pas sûr du type de demande."}

# Tests
tests = [
    "Quels sont les délais de livraison ?",
    "Je cherche une chemise bleu marine pour homme (casual) en hiver, budget 50",
    "Propose un look pour une soirée",
    "Quelle différence entre un jean et un chino ?"
]

for t in tests:
    out = assistant(t, model="llama3:latest")
    print("\n---")
    print("Q:", t)
    print("Route:", out.get("route"), "| Type:", out.get("task_type"))
    print(out.get("answer", "")[:400])

#  C- RAG complet (embeddings + index + retrieval)

#### C.1 Transformer en *documents* (texte + metadata)
---
-  --- Test end-to-end (filters -> candidates -> context -> answer) ---

On fabrique un format unifié, Chaque document contient :
```python
{
  "id": "prod:15970",
  "text": "...",
  "meta": {"type":"product", "role":"public", ...}
}
```

In [19]:
DOCS = []

# Produits
for p in PRODUCTS:
    pid = p.get("product_id", p.get("id", "NA"))
    doc_id = f"prod:{pid}"

    price = p.get("price", "N/A")

    text = (
        f"ProductDisplayName: {p.get('productDisplayName','')}\n"
        f"Gender: {p.get('gender','')}\n"
        f"MasterCategory: {p.get('masterCategory','')}\n"
        f"SubCategory: {p.get('subCategory','')}\n"
        f"ArticleType: {p.get('articleType','')}\n"
        f"BaseColour: {p.get('baseColour','')}\n"
        f"Season: {p.get('season','')}\n"
        f"Year: {p.get('year','')}\n"
        f"Usage: {p.get('usage','')}\n"
        f"Price: {price}\n"
    )

    DOCS.append({
        "id": doc_id,
        "text": text,
        "meta": {
            "type": "product",
            "role": "public",
            "product_id": pid,
            "gender": p.get("gender"),
            "articleType": p.get("articleType"),
            "baseColour": p.get("baseColour"),
            "season": p.get("season"),
            "usage": p.get("usage"),
        }
    })

# FAQ
for i, f in enumerate(FAQ_DATA):
    DOCS.append({
        "id": f"faq:{i}",
        "text": f"Question: {f.get('question','')}\nAnswer: {f.get('answer','')}",
        "meta": {"type": "faq", "role": "public"}
    })

print("✅ DOCS créés :", len(DOCS))
print("Exemple DOC:", DOCS[0]["id"])
print(DOCS[0]["text"][:350])
print("Meta preview:", DOCS[0]["meta"])

✅ DOCS créés : 44449
Exemple DOC: prod:15970
ProductDisplayName: Turtle Check Men Navy Blue Shirt
Gender: Men
MasterCategory: Apparel
SubCategory: Topwear
ArticleType: Shirts
BaseColour: Navy Blue
Season: Fall
Year: 2011.0
Usage: Casual
Price: 67

Meta preview: {'type': 'product', 'role': 'public', 'product_id': 15970, 'gender': 'Men', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'season': 'Fall', 'usage': 'Casual'}


-   `🟦 À retenir`
    -   Le RAG travaille sur des documents, pas des lignes de DataFrame
    -   Les métadonnées permettent sécurité et filtrage

#### C.2- Test embeddings sur un petit échantillon

In [20]:
EMBED_MODEL = "nomic-embed-text"

sample = DOCS[:30]
t0 = time.perf_counter()

vecs = []
for d in sample:
    v, _ = utils.ollama_embed(d["text"], model=EMBED_MODEL)
    vecs.append(v)

DOC_EMB_SAMPLE = np.array(vecs, dtype=np.float32)
print("✅ sample embeddings shape:", DOC_EMB_SAMPLE.shape)
print("⏱️ sample time:", int((time.perf_counter()-t0)*1000), "ms")

✅ sample embeddings shape: (30, 768)
⏱️ sample time: 1554 ms


#### C.3- Embeddings (Ollama) + Index vectoriel local

- Embeddings : `nomic-embed-text`
- Index : `NearestNeighbors` (cosine) — très simple et 100% local

> Pour un dataset très grand, on passera ensuite à FAISS/Chroma. Ici on veut un lab *stable*.


- Embeddings COMPLETS + cache (docs.jsonl + embeddings.npy + nn_index.joblib)

In [21]:
CACHE_DIR = "data"
os.makedirs(CACHE_DIR, exist_ok=True)

DOCS_PATH = os.path.join(CACHE_DIR, "docs.jsonl")
X_PATH    = os.path.join(CACHE_DIR, "embeddings.npy")
NN_PATH   = os.path.join(CACHE_DIR, "nn_index.joblib")
META_PATH = os.path.join(CACHE_DIR, "cache_meta.json")

def save_docs_jsonl(docs, path):
    with open(path, "w", encoding="utf-8") as f:
        for d in docs:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def load_docs_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            out.append(json.loads(line))
    return out

def save_meta(meta, path=META_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

def load_meta(path=META_PATH):
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def cache_is_compatible(meta, docs_len_expected):
    if not meta:
        return False
    return (
        meta.get("embed_model") == EMBED_MODEL
        and meta.get("n_docs") == docs_len_expected
        and isinstance(meta.get("dim"), int)
    )

def build_or_load(docs, force_rebuild=False, k_neighbors=8, progress_every=500):
    docs_len = len(docs)
    meta = load_meta()

    if (not force_rebuild) and os.path.exists(DOCS_PATH) and os.path.exists(X_PATH) and os.path.exists(NN_PATH) and cache_is_compatible(meta, docs_len):
        print("✅ Cache compatible trouvé, chargement...")
        d2 = load_docs_jsonl(DOCS_PATH)
        X2 = np.load(X_PATH).astype(np.float32, copy=False)
        nn2 = joblib.load(NN_PATH)
        print("✅ Chargé:", len(d2), X2.shape, "| embed_model:", meta.get("embed_model"))
        return d2, X2, nn2

    print("⚙️ Pas de cache compatible → Calcul embeddings (peut prendre du temps)...")
    texts = [d["text"] for d in docs]
    vecs = []
    t0 = time.perf_counter()

    for i, t in enumerate(texts, start=1):
        v, _ = utils.ollama_embed(t, model=EMBED_MODEL)
        vecs.append(v)

        if i % progress_every == 0:
            elapsed = time.perf_counter() - t0
            rate = i / max(elapsed, 1e-6)
            remaining = (docs_len - i) / max(rate, 1e-6)
            print(f"  ... {i}/{docs_len} ({(i/docs_len)*100:.1f}%) | ~{int(remaining)}s restantes")

    X = np.array(vecs, dtype=np.float32)
    total_ms = int((time.perf_counter()-t0)*1000)
    print("✅ Embeddings:", X.shape, "| total:", total_ms, "ms")

    nn = NearestNeighbors(n_neighbors=k_neighbors, metric="cosine")
    nn.fit(X)
    print("✅ Index prêt.")

    # Save cache
    save_docs_jsonl(docs, DOCS_PATH)
    np.save(X_PATH, X)
    joblib.dump(nn, NN_PATH)

    save_meta({"embed_model": EMBED_MODEL, "n_docs": docs_len, "dim": int(X.shape[1])})
    print("💾 Cache sauvegardé (+ meta).")

    return docs, X, nn

DOCS, DOC_EMB, NN_INDEX = build_or_load(DOCS, force_rebuild=False, k_neighbors=8)

print("✅ DOCS total:", len(DOCS))
print("✅ DOC_EMB shape:", DOC_EMB.shape)
print("✅ NN_INDEX ready:", NN_INDEX is not None)

⚙️ Pas de cache compatible → Calcul embeddings (peut prendre du temps)...
  ... 500/44449 (1.1%) | ~1001s restantes
  ... 1000/44449 (2.2%) | ~993s restantes
  ... 1500/44449 (3.4%) | ~961s restantes
  ... 2000/44449 (4.5%) | ~940s restantes
  ... 2500/44449 (5.6%) | ~919s restantes
  ... 3000/44449 (6.7%) | ~906s restantes
  ... 3500/44449 (7.9%) | ~893s restantes
  ... 4000/44449 (9.0%) | ~879s restantes
  ... 4500/44449 (10.1%) | ~867s restantes
  ... 5000/44449 (11.2%) | ~856s restantes
  ... 5500/44449 (12.4%) | ~845s restantes
  ... 6000/44449 (13.5%) | ~837s restantes
  ... 6500/44449 (14.6%) | ~829s restantes
  ... 7000/44449 (15.7%) | ~814s restantes
  ... 7500/44449 (16.9%) | ~799s restantes
  ... 8000/44449 (18.0%) | ~784s restantes
  ... 8500/44449 (19.1%) | ~771s restantes
  ... 9000/44449 (20.2%) | ~762s restantes
  ... 9500/44449 (21.4%) | ~751s restantes
  ... 10000/44449 (22.5%) | ~743s restantes
  ... 10500/44449 (23.6%) | ~732s restantes
  ... 11000/44449 (24.7%) | ~

-   `🟦 À retenir`
    -   Le cache est indispensable en production
    -   L’index est indépendant du LLM de génération

#### C.4- Retrieval (+ RBAC filter)
--- 
- récupérer les documents pertinents en respectant les rôles.

On récupère top-k docs puis on filtre selon le rôle utilisateur :
- `user_role="public"` → interdit d’utiliser docs `role="staff"`
- `user_role="staff"` → autorisé


In [22]:
import time

SYSTEM = (
    "Tu es un assistant utile. "
    "Réponds UNIQUEMENT en utilisant le contexte fourni. "
    "Chaque phrase factuelle DOIT se terminer par au moins une citation entre crochets "
    "comme [prod:123] ou [faq:4]. "
    "N'invente jamais de devise (€, $, FCFA) ni de prix : réutilise exactement la valeur 'Price' du contexte. "
    "Si le contexte ne contient pas la réponse, dis exactement : "
    "Je n'ai pas assez d'informations dans les documents fournis."
)

def build_context(hits, max_chars: int = 3500) -> str:
    parts, used = [], 0
    for h in hits:
        d = h["doc"]
        block = f"[{d['id']}]\n{d['text']}\n---\n"
        if used + len(block) > max_chars:
            break
        parts.append(block)
        used += len(block)
    return "".join(parts).strip()

def _rbac_allow(meta: dict, user_role: str) -> bool:
    role = (meta or {}).get("role", "public")
    return not (role == "staff" and user_role != "staff")

def retrieve(query: str, k: int = 8, user_role: str = "public"):
    """
    Retourne: (hits, q_embed_ms, rbac_stats)
    hits = [{"doc": DOCS[i], "distance": float(dist)}]
    """
    k = max(1, int(k))
    if len(DOCS) == 0:
        return [], 0, {"n_before_rbac": 0, "n_blocked": 0}

    qvec, qms = utils.ollama_embed(query, model=EMBED_MODEL)
    q = np.array(qvec, dtype=np.float32).reshape(1, -1)

    # On prend un peu plus large pour compenser le filtrage RBAC
    k0 = min(max(k * 3, k), len(DOCS))
    distances, indices = NN_INDEX.kneighbors(q, n_neighbors=k0)

    hits = []
    blocked = 0
    for dist, idx in zip(distances[0], indices[0]):
        d = DOCS[int(idx)]
        if _rbac_allow(d.get("meta", {}), user_role):
            hits.append({"doc": d, "distance": float(dist)})
        else:
            blocked += 1
        if len(hits) >= k:
            break

    stats = {"n_before_rbac": int(k0), "n_blocked": int(blocked)}
    return hits, qms, stats

def answer_rag(
    query: str,
    k: int = 8,
    user_role: str = "public",
    model: str = "llama3:latest",
    temperature: float = 0.2
) -> dict:
    t0 = time.perf_counter()

    hits, qms, rbac_stats = retrieve(query, k=k, user_role=user_role)
    context = build_context(hits)

    user_prompt = (
        f"Contexte :\n{context}\n\n"
        f"Question : {query}\n\n"
        "Consignes:\n"
        "- Réponds en français.\n"
        "- Chaque phrase factuelle doit contenir au moins une citation [prod:...] ou [faq:...].\n"
        "- Si tu ne peux pas répondre avec le contexte, dis la phrase exacte demandée.\n"
    )

    out = utils.ollama_chat(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": user_prompt}],
        model=model,
        temperature=temperature
    )

    latency_ms_total = int((time.perf_counter() - t0) * 1000)

    return {
        "type": "rag",
        "query": query,
        "k": int(k),
        "user_role": user_role,
        "model": model,
        "answer": out.text,
        "context": context,
        "retrieved": [
            {
                "id": h["doc"]["id"],
                "distance": h["distance"],
                "role": h["doc"].get("meta", {}).get("role", "public"),
                "type": h["doc"].get("meta", {}).get("type", "unknown"),
            }
            for h in hits
        ],
        "q_embed_ms": int(qms),
        "llm_latency_ms": int(out.latency_ms),
        "latency_ms_total": latency_ms_total,
        "rbac": rbac_stats,
        "prompt_chars": len(user_prompt),
    }

-   `🟦 À retenir`
    -   La sécurité doit être appliquée avant le prompt
    -   Le LLM ne doit jamais voir ce qu’il n’a pas le droit de voir

- Mini test

In [23]:
res = answer_rag("Je cherche une chemise bleu marine pour homme", k=8, user_role="public", model="llama3:latest")
print(res["answer"][:400])
print("retrieved:", len(res["retrieved"]), "| blocked:", res["rbac"]["n_blocked"])

Je suis désolé, mais il n'y a pas de chemise bleu marine pour homme dans les produits fournis.
retrieved: 8 | blocked: 0


#### C.5- Prompt RAG strict + génération (Ollama chat)
---
- forcer des réponses fiables.

Règles :
- répondre **uniquement** à partir du contexte récupéré
- citations **obligatoires** : `[doc_id]`
- phrase de fallback exacte: si pas d’info : dire **“Je ne sais pas d’après les documents fournis.”**


- Construire le prompt RAG.

In [24]:
def build_rag_prompt(query: str, context: str) -> list:
    """
    Retourne une liste de messages (format chat) pour Ollama.
    """
    system = SYSTEM

    user = (
        f"Contexte :\n{context}\n\n"
        f"Question : {query}\n\n"
        "Consignes:\n"
        "- Réponds en français.\n"
        "- Chaque phrase factuelle doit contenir au moins une citation [prod:...] ou [faq:...].\n"
        "- N'invente pas d'informations absentes du contexte.\n"
        "- Si tu ne peux pas répondre avec le contexte, dis exactement :\n"
        "Je n'ai pas assez d'informations dans les documents fournis.\n"
    )

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

In [25]:
# Preview
msgs = build_rag_prompt("Test question", "[prod:1]\nPrice: 10\n---")
print(msgs[1]["content"][:600])

Contexte :
[prod:1]
Price: 10
---

Question : Test question

Consignes:
- Réponds en français.
- Chaque phrase factuelle doit contenir au moins une citation [prod:...] ou [faq:...].
- N'invente pas d'informations absentes du contexte.
- Si tu ne peux pas répondre avec le contexte, dis exactement :
Je n'ai pas assez d'informations dans les documents fournis.



-   `🟦 À retenir`
    -   Le prompt est un contrat
    -   Plus il est strict, moins il y a d’hallucinations

#### C.6- Génération depuis le prompt
---
-   Appel Ollama
-   Mesure de latence

In [26]:
def generate_answer_from_prompt(messages, model="llama3:latest", temperature=0.2) -> dict:
    out = utils.ollama_chat(messages, model=model, temperature=temperature)
    return {"text": out.text, "llm_latency_ms": out.latency_ms}

In [27]:
# Test rapide
gen = generate_answer_from_prompt(msgs, model="llama3:latest")
print(gen["text"][:300])
print("llm_latency_ms:", gen["llm_latency_ms"])

Je suis désolé, mais je n'ai pas suffisamment d'informations pour répondre à votre question. Le contexte ne contient que des informations sur un produit [prod:1] avec un prix de 10, mais il n'y a pas de réponse à la question "Test question".
llm_latency_ms: 6064


In [ ]:
res = answer_rag(
    "Avez-vous une chemise bleu marine pour homme (usage casual) ?",
    k=8,
    user_role="public",
    model="llama3:latest"
)
print(res["answer"])
print("\nlatency:", res["latency_ms_total"], "ms | retrieved:", len(res["retrieved"]))

# D-Production mindset

#### D.1- Observabilité : traces JSONL
--- 
- Traces compactes
- Pas de contexte lourd

On loggue chaque requête dans `logs/traces.jsonl`.


In [29]:
from pathlib import Path
import json, time, uuid

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)
TRACE_FILE = LOG_DIR / "traces.jsonl"

def build_trace(res: dict) -> dict:
    """
    Construit une trace compacte et lisible pour le monitoring.
    """
    return {
        "trace_id": str(uuid.uuid4()),
        "timestamp_ms": int(time.time() * 1000),

        # Input
        "query": res.get("query"),
        "user_role": res.get("user_role"),
        "k": res.get("k"),
        "model": res.get("model"),

        # Retrieval
        "n_retrieved": len(res.get("retrieved", [])),
        "retrieved_ids": [h["id"] for h in res.get("retrieved", [])],
        "rbac": res.get("rbac"),

        # Generation
        "answer_chars": len(res.get("answer", "")),
        "llm_latency_ms": res.get("llm_latency_ms"),
        "total_latency_ms": res.get("latency_ms_total"),

        # Prompt / context
        "prompt_chars": res.get("prompt_chars"),

        # Meta
        "type": res.get("type", "rag"),
    }

def log_trace(trace: dict):
    with TRACE_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(trace, ensure_ascii=False) + "\n")

# --- Log de la dernière requête ---
trace = build_trace(res)
log_trace(trace)

print("✅ Trace loggée ->", TRACE_FILE)
print("Trace preview:")
print(json.dumps(trace, indent=2, ensure_ascii=False))


✅ Trace loggée -> logs/traces.jsonl
Trace preview:
{
  "trace_id": "48479253-c8dd-48f6-8258-aff5e7f8946d",
  "timestamp_ms": 1770659843793,
  "query": "Je cherche une chemise bleu marine pour homme",
  "user_role": "public",
  "k": 8,
  "model": "llama3:latest",
  "n_retrieved": 8,
  "retrieved_ids": [
    "prod:43963",
    "prod:32563",
    "prod:43962",
    "prod:46762",
    "prod:23601",
    "prod:46763",
    "prod:46755",
    "prod:45685"
  ],
  "rbac": {
    "n_before_rbac": 24,
    "n_blocked": 0
  },
  "answer_chars": 94,
  "llm_latency_ms": 7392,
  "total_latency_ms": 7518,
  "prompt_chars": 2242,
  "type": "rag"
}


-   `🟦 À retenir`
    -   En production, on ne loggue pas tout
    -   On loggue ce qui permet de comprendre et déboguer

- Vérification rapide

In [ ]:
with open("logs/traces.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()[-3:]

for l in lines:
    print(json.loads(l))


#### D.2- Évaluation simple (sans LLM-judge)
---

Métriques :
1. **Couverture de citations** : % des phrases qui ont au moins un `[doc_id]`
2. **Chevauchement réponse↔contexte** : overlap de tokens (proxy de “fidélité au contexte”)
3. **Pertinence de la recherche** : similarité query ↔ top docs (cosine via embeddings)


In [30]:
import re

def split_sentences(text: str):
    s = re.split(r'(?<=[.!?])\s+', (text or "").strip())
    return [x for x in s if x]

def is_factual_sentence(s: str, min_tokens: int = 5) -> bool:
    tokens = re.findall(r"[A-Za-z0-9À-ÿ]+", s)
    return len(tokens) >= min_tokens

def citation_coverage(answer: str) -> float:
    sents = [s for s in split_sentences(answer) if is_factual_sentence(s)]
    if not sents:
        return 1.0  # pas de phrase factuelle → pas de violation
    with_cite = sum(1 for s in sents if re.search(r'\[[^\]]+\]', s))
    return with_cite / len(sents)

def token_overlap(answer: str, context: str) -> float:
    tok = lambda t: set(re.findall(r"[A-Za-z0-9À-ÿ]+", (t or "").lower()))
    A = tok(answer)
    C = tok(context)
    if not A or not C:
        return 0.0
    return len(A & C) / len(A)

def retrieval_relevance_from_retrieved(retrieved: list) -> float:
    # relevance ≈ moyenne de (1 - cosine_distance)
    if not retrieved:
        return 0.0
    rels = [max(0.0, 1.0 - float(h.get("distance", 1.0))) for h in retrieved]
    return float(np.mean(rels))

metrics = {
    "citation_coverage": round(citation_coverage(res["answer"]), 3),
    "context_overlap": round(token_overlap(res["answer"], res.get("context", "")), 3),
    "retrieval_relevance": round(retrieval_relevance_from_retrieved(res.get("retrieved", [])), 3),
}

metrics

{'citation_coverage': 0.0,
 'context_overlap': 0.053,
 'retrieval_relevance': 0.556}

#### D.3- Compromis qualité/latence : comparer 3 configs (k + modèle)
---

On compare :
- `k=3` vs `k=8`
- modèle fast vs modèle quality (si installé)
- Score global


In [31]:
import pandas as pd
import numpy as np

CHAT_MODEL_FAST = "gemma2:2b"
CHAT_MODEL_QUAL = "gemma3:12b"

def ollama_has_model(name: str) -> bool:
    # vérif simple via /api/tags (si tu veux éviter web/curl)
    try:
        import urllib.request, json
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        models = [m.get("name") for m in data.get("models", [])]
        return name in models
    except Exception:
        return True  # si on ne peut pas vérifier, on tente quand même

def overall_score(cite_cov, ctx_overlap, retr_rel):
    # Score simple (0..1 environ), pondérations explicables
    # - citations (0.5)
    # - overlap (0.2)
    # - relevance (0.3)
    return 0.5*cite_cov + 0.2*ctx_overlap + 0.3*retr_rel

def run_tradeoff_suite(query: str, user_role: str = "public"):
    configs = [
        {"k": 3, "model": CHAT_MODEL_FAST},
        {"k": 8, "model": CHAT_MODEL_FAST},
    ]
    if ollama_has_model(CHAT_MODEL_QUAL):
        configs.append({"k": 8, "model": CHAT_MODEL_QUAL})

    rows = []
    for cfg in configs:
        r = answer_rag(query, k=cfg["k"], user_role=user_role, model=cfg["model"])

        cite_cov = citation_coverage(r["answer"])
        ctx_ov = token_overlap(r["answer"], r.get("context",""))
        retr_rel = retrieval_relevance_from_retrieved(r.get("retrieved", []))

        m = {
            "k": cfg["k"],
            "model": cfg["model"],
            "latency_ms_total": r["latency_ms_total"],
            "llm_latency_ms": r.get("llm_latency_ms"),
            "cite_cov": round(cite_cov, 3),
            "ctx_overlap": round(ctx_ov, 3),
            "retr_rel": round(retr_rel, 3),
            "score": round(overall_score(cite_cov, ctx_ov, retr_rel), 3),
        }

        rows.append(m)
        log_trace({"type": "tradeoff", "result": r, "metrics": m})

    df = pd.DataFrame(rows)
    return df.sort_values(["score", "latency_ms_total"], ascending=[False, True])

df_trade = run_tradeoff_suite(
    "Je cherche une montre argentée pour femme (hiver), budget 50. As-tu des options ?",
    user_role="public"
)
df_trade

,k,model,latency_ms_total,llm_latency_ms,cite_cov,ctx_overlap,retr_rel,score
0,3,gemma2:2b,2934,2257,0.0,0.0,0.589,0.177
1,8,gemma2:2b,1116,864,0.0,0.0,0.579,0.174
2,8,gemma3:12b,14003,13907,0.0,0.0,0.579,0.174


-   `🟦 À retenir`
    -   Plus de qualité = souvent plus de latence
    -   Le “meilleur” modèle dépend du contexte

#### D.4- Sécurité : démo RBAC (Role-Based Access Control)

Même question, deux rôles :
- `public` : ne doit jamais utiliser des docs `staff`
- `staff` : autorisé
- Injection docs staff
- Comparaison public vs staff


##### a- Injecter 3 docs “staff_only” (une seule fois)

In [32]:
staff_docs = [
    {"id":"staff:promo1", "text":"Internal promo list: some items priced at 2 are staff-only.", "meta":{"role":"staff", "type":"internal"}},
    {"id":"staff:stock1", "text":"Internal stock note: navy blue shirts - only 2 left (staff-only).", "meta":{"role":"staff", "type":"internal"}},
    {"id":"staff:marge1", "text":"Internal margin note: silver watches - margin 35% (staff-only).", "meta":{"role":"staff", "type":"internal"}},
]

# On évite de ré-injecter si déjà fait
existing_ids = set(d["id"] for d in DOCS)
to_add = [d for d in staff_docs if d["id"] not in existing_ids]

if to_add:
    DOCS.extend(to_add)

    new_vecs = []
    for d in to_add:
        v, _ = utils.ollama_embed(d["text"], model=EMBED_MODEL)
        new_vecs.append(v)

    new_vecs = np.array(new_vecs, dtype=np.float32)
    DOC_EMB = np.vstack([DOC_EMB, new_vecs])

    # Refit index (rapide car on a juste ajouté 3 lignes)
    NN_INDEX = NearestNeighbors(n_neighbors=8, metric="cosine").fit(DOC_EMB)

print("✅ Staff docs ajoutés:", len(to_add), "| total DOCS:", len(DOCS))

✅ Staff docs ajoutés: 3 | total DOCS: 44452


##### b- Test RBAC

In [33]:
q = "Any very cheap items under 5 dollars?"

res_public = answer_rag(q, k=8, user_role="public", model=CHAT_MODEL_FAST)
res_staff  = answer_rag(q, k=8, user_role="staff",  model=CHAT_MODEL_FAST)

print("PUBLIC retrieved roles:", sorted(set([h["role"] for h in res_public["retrieved"]])))
print(res_public["answer"][:400], "...\n")

print("STAFF retrieved roles:", sorted(set([h["role"] for h in res_staff["retrieved"]])))
print(res_staff["answer"][:400], "...\n")

PUBLIC retrieved roles: ['public']
Je n'ai pas assez d'informations dans les documents fournis. 
 ...

STAFF retrieved roles: ['public', 'staff']
Je n'ai pas assez d'informations dans les documents fournis. 
 ...



# E- Déploiement API: Chatbot RAG via Flask
---
-   /health
-   /ask
-   Chargement cache au démarrage
-   Logs automatiques